In [1]:
from train import MusicLSTM
from torch.nn.functional import softmax
import pickle as pkl
import torch

In [2]:
with open('itos.pkl','rb') as f:
    itos = pkl.load(f)

with open('stoi.pkl','rb') as f:
    stoi = pkl.load(f)

vocab_size = len(itos)

Model = MusicLSTM(vocab_size,64,256,2,0.4)
Model.load_state_dict(
    torch.load("music_lstm_best.pth", weights_only=False, map_location='cpu')
)



<All keys matched successfully>

In [3]:
Model.eval()

MusicLSTM(
  (embed): Embedding(62, 64)
  (lstm): LSTM(64, 256, num_layers=2, batch_first=True, dropout=0.4)
  (linear): Linear(in_features=256, out_features=62, bias=True)
)

In [ ]:
from music21 import converter

def generate_tune(model, stoi, itos, max_len=1000, temperature=1.0):
    model.eval()
    X = stoi['`']
    output = ''
    hidden = None
    
    with torch.no_grad():
        while len(output) < max_len:
            X = torch.tensor([[X]]) 
            logit, hidden = model(X, hidden)
            prob = torch.softmax(logit.squeeze() / temperature, dim=-1)
            next_token = torch.multinomial(prob, num_samples=1).item()
            if next_token == stoi['$']:
                break
            output += itos[next_token]
            X = next_token
    
    return output

for attempt in range(10):
    tune = generate_tune(Model, stoi, itos)
    try:
        score = converter.parse(tune, format='abc')
        score.write('midi', 'generated.mid')
        print(f'Success on attempt {attempt+1}')
        break
    except:
        print(f'Attempt {attempt+1} failed, retrying...')

Attempt 1 failed, retrying...
Attempt 2 failed, retrying...
Attempt 3 failed, retrying...
Attempt 4 failed, retrying...
Attempt 5 failed, retrying...
Success on attempt 6


In [15]:
Model.eval()

with open('whistle.abc','r') as f:
    start = f.read()
print(start)

M 4/4
L: 1/8
Q:1/4=120
K:Eb
z/2E,,/2z4z3/2d'z/2| \
z4 z3/2e'e'/2z| \
z3z/2d'/2 z4| \
z4 z_d' 



In [18]:
init = [stoi[char] for char in start]
init = [stoi['~']] + init
init

[30,
 9,
 41,
 14,
 22,
 14,
 23,
 59,
 10,
 41,
 3,
 22,
 8,
 23,
 57,
 10,
 3,
 22,
 14,
 53,
 3,
 37,
 6,
 23,
 56,
 10,
 60,
 52,
 23,
 16,
 22,
 37,
 60,
 45,
 45,
 22,
 37,
 16,
 14,
 16,
 1,
 22,
 37,
 36,
 25,
 16,
 22,
 37,
 26,
 41,
 58,
 23,
 16,
 14,
 41,
 16,
 1,
 22,
 37,
 12,
 25,
 12,
 25,
 22,
 37,
 16,
 26,
 41,
 58,
 23,
 16,
 1,
 16,
 22,
 37,
 36,
 25,
 22,
 37,
 41,
 16,
 14,
 26,
 41,
 58,
 23,
 16,
 14,
 41,
 16,
 24,
 36,
 25,
 41,
 23]

In [32]:
hidden = None
output = start

with torch.no_grad():
    for X in init[:-1]:
          X = torch.tensor([[X]])
          logit, hidden = Model(X, hidden)

In [33]:
X = init[-1]
with torch.no_grad():
    while True:
         X = torch.tensor([[X]]) 
         logit, hidden = Model(X, hidden)
         prob = torch.softmax(logit.squeeze() , dim=-1)
         next_token = torch.multinomial(prob, num_samples=1).item()
         if next_token == stoi['$']:
            break
         output += itos[next_token]
         X = next_token

In [26]:
output

'M 4/4\nL: 1/8\nQ:1/4=120\nK:Eb\nz/2E,,/2z4z3/2d\'z/2| \\\nz4 z3/2e\'e\'/2z| \\\nz3z/2d\'/2 z4| \\\nz4 z_d\' \n"D"f3/2a/2|g3|"D7"c2d f/2ga/2a|"G"g2f e2:|[2"C"e2e e2c|"G"d2g dBc|\n"G"d2d ded|"C"c2d "D7"e2f|"G"g2g gfe|"Am"c2e "D7"d3|"G"A2G B2c|\n"G"dcd "G7"d=cB|"Dm"ABA "G"GAB|"C"cde "G"dcB|"Am"A3 "D7"ADG|"Dm"f3 "G"g3|\n"C"e2g efg|"G/d"g2g "C"b2a|"G"gBg g2g|"G"d^cB "D7"AFG|\n"G"GAB dBG|"C"c3 efg|"G"ded dBG|"C"CEG "G"G2d|\n"C"eed "C"cec|"G"BAG "D7"AFA|"G"GGB "D7"cBd|\n"G"b2a "C"gfe|"G"dbd "C"ecA|"D7"dGD "G"G2:|'

In [30]:
from music21 import converter


In [34]:
from music21 import converter, midi

score = converter.parse(output, format='abc')
score.write('midi', 'generated_from_whistle.mid')

'generated_from_whistle.mid'

In [ ]:
from midi2audio import FluidSynth
fs = FluidSynth('/usr/share/soundfonts/FluidR3_GM.sf2')
fs.midi_to_audio('generated.mid', 'generated.wav')

ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) [error.pcm] unable to open slave
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) [error.pcm] Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) [error.pcm] Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) [error.pcm] Unknown PCM cards.pcm.side
ALSA lib pcm_route.c:886:(find_matching_chmap) [error.pcm] Found no matching channel map
ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) [error.pcm] unable to open slave
error: '-F' is an illegal option at this place, only -b option is allowed here.
fluidsynth: error: fluid_is_soundfont(): fopen() failed: 'File does not exist.'
Parameter 'generated.wav' not a SoundFont or MIDI file or error occurred identifying it.
error: '-r' is an illegal option at this place, only -b option is allowed here.
fluidsynth: error: fluid_is_soundfont(): fopen() failed: 'File does not exist.'
Parameter '44100' not a SoundFont or MIDI file or error occurred identifying it.
fluidsynth

FluidSynth runtime version 2.5.4
Copyright (C) 2000-2026 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

